In [1]:
import os
import logging
import numpy as np
import openai
openai.api_key = os.environ["OPENAI_API_KEY"]

from tenacity import (retry, stop_after_attempt,  # for exponential backoff
                      wait_random_exponential)

In [2]:
# general definitions and helper functions

# SPLITTER = '\n'
SPLITTER = '.'

# generations per question
n_regenerate = 3
n_questions = 'two'
n_facts = 'four'
n_repeat_question_gen = 2

@retry(wait=wait_random_exponential(min=3, max=20), stop=stop_after_attempt(100))
def oai_predict(prompt):
    if isinstance(prompt, str):
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": prompt},
        ]
    else:
        messages = prompt
    
    output = openai.ChatCompletion.create(
        model='gpt-3.5-turbo',
        # model='gpt-4',
        messages=messages,
        max_tokens=200,
    )
    response = output['choices'][0]['message']['content']
    return response


def log_w_indent(text, indent):
    logging.info((indent * 2) * '>>' + ' ' + text)

def predict_w_log(prompt, indent):
    log_w_indent(f'Input: {prompt}', indent)
    response = oai_predict(prompt)
    log_w_indent(f'Output: {response}', indent)
    return response

def predict_wo_log(prompt, indent):
    response = oai_predict(prompt)
    return response

def setup_logger():
    """Setup logger to always print time and level."""
    logging.basicConfig(
        format='%(asctime)s %(levelname)-8s %(message)s',
        level=logging.INFO,
        datefmt='%Y-%m-%d %H:%M:%S')
    logging.getLogger().setLevel(logging.INFO)  # logging.DEBUG
setup_logger()

def divider(symbol = '*'):
    logging.info(80 * symbol)


base_initial_prompt = "Respond with " + n_facts + " short sentences to the following question. Provide concrete facts rather than vague descriptions.\n{user_question}"

base_gen_questions_prompt = 'In the context of the question "{user_question}" please generate a numbered list of ' + n_questions + ' questions that might have the answer "{fact}"\nYour questions should avoid using specific facts in the answer, but can be specific to the context implied by the original question. Your questions should be so specific that only one answer is possible. You can ask questions starting with "What", "Where", or "Who" if appropriate.'
base_answer_question_prompt = 'In the context of "{user_question}" respond as concisely as possible to the following question: "{question}"'

base_equivalence_prompt = '"{fact}" is the proposed answer to the question "{regen_question}" in the context of the larger question "{user_question}".\nWe are trying to decide if the proposed true is accurate by comparing it to a number of brainstormed alternative answers:'
for i in range(1, n_regenerate + 1):
    base_equivalence_prompt += f'\nAlternative Answer {i}: ' + '{}'
base_equivalence_prompt += '\nRespond only with "yes" or "no".  Is the proposed answer "{fact}" compatible with the alternative answers 1-' + str(n_regenerate) + '?'

In [3]:
results = dict()
results['prompts'] = dict(
    base_initial_prompt = base_initial_prompt,
    base_gen_questions_prompt = base_gen_questions_prompt,
    base_answer_question_prompt = base_answer_question_prompt,
    base_equivalence_prompt = base_equivalence_prompt,
)
logging.info(f'Using prompts {results["prompts"]}')

2023-09-26 16:47:43 INFO     Using prompts {'base_initial_prompt': 'Respond with four short sentences to the following question. Provide concrete facts rather than vague descriptions.\n{user_question}', 'base_gen_questions_prompt': 'In the context of the question "{user_question}" please generate a numbered list of two questions that might have the answer "{fact}"\nYour questions should avoid using specific facts in the answer, but can be specific to the context implied by the original question. Your questions should be so specific that only one answer is possible. You can ask questions starting with "What", "Where", or "Who" if appropriate.', 'base_answer_question_prompt': 'In the context of "{user_question}" respond as concisely as possible to the following question: "{question}"', 'base_equivalence_prompt': '"{fact}" is the proposed answer to the question "{regen_question}" in the context of the larger question "{user_question}".\nWe are trying to decide if the proposed true is accu

In [4]:
# Ilya Sutskever
# Yann LeCun
# Pieter Abbeel
# Fei-Fei Li
# Chelsea Finn
# Dawn Song
# Zhang Tong
# Tim Rocktäschel
# Anca Dragan
# George Konidaris

user_questions = ['Who is Yarin Gal?']
entities = ['YG']

wait = lambda: input('wait')
# wait = lambda: 0

for user_question, entity in zip(user_questions, entities):
    # << ASK USER QUESTION >>
    divider()
    log_w_indent(f'Starting with entity {entity}, user_question: "{user_question}"', 0)
    
    results[entity] = {'user_question': user_question, 'entity': entity}
    e_results = results[entity]

    intitial_prompt = base_initial_prompt.format(user_question=user_question)
    e_results['initial'] = predict_w_log(intitial_prompt, 1)
    
    # set up new results
    e_results['final'] = []

    # << SPLIT INTO FACTS >>
    
    # split response into facts
    facts = [(r + SPLITTER) for r in e_results['initial'].replace('Ph.D.', 'PhD').split(SPLITTER) if r]
    facts = [f.replace('\n', '').strip() for f in facts]

    # let's add a hallucination 
    for i, fact in enumerate(facts):
        if 'Oxford' in fact:
            facts[i] = fact.replace('Oxford', 'Cambridge')
            logging.warning('Adding hallucination manually!')
    
    # fill in later
    e_results['facts'] = facts
    log_w_indent(f'Extracted facts: {facts}', 1)

    e_results['regen_questions'] = {}
    e_results['regen_answers'] = {}
    e_results['regen_compatible'] = {}
    e_results['uncertainty'] = {}

    wait()
    # << ATTEND TO EACH FACT SEPARATELY >>
    for fidx, fact in enumerate(facts):
        # << GENERATE QUESTIONS ABOUT FACT >>
        log_w_indent(f'Currently dealing with fact {fidx}: {fact}', 2)

        e_results['regen_answers'][f'fact-{fidx}'] = {}
        e_results['regen_compatible'][f'fact-{fidx}'] = {}
        e_results['uncertainty'][f'fact-{fidx}'] = {}

        # << GENERATE ANSWERS FOR EACH QUESTION >>

        questions = []
        for it in range(n_repeat_question_gen):
            gen_questions = predict_w_log(base_gen_questions_prompt.format(user_question=user_question, fact=fact), 3)    
            questions.extend([q[3:] for q in gen_questions.split('\n') if q])

        log_w_indent(f'Extracted questions: {questions}', 2)
        e_results['regen_questions'][f'fact-{fidx}'] = {}
        e_results['regen_answers'][f'fact-{fidx}'] = {}
        e_results['regen_compatible'][f'fact-{fidx}'] = {}
        e_results['uncertainty'][f'fact-{fidx}'] = {}
        wait()
        for qidx, question in enumerate(questions):
            log_w_indent(f'Regenerate answers for question {qidx} "{question}":', 3)

            e_results['regen_answers'][f'fact-{fidx}'][f'question-{qidx}'] = []
            regen_answers = e_results['regen_answers'][f'fact-{fidx}'][f'question-{qidx}']

            # << ANSWER EACH QUESTION MULTIPLE TIMES >>
            for re_gen in range(n_regenerate):
                answer = predict_w_log(base_answer_question_prompt.format(
                    user_question=user_question, question=question), 4)
                regen_answers.append(answer)

            # << CHECK IF ANSWERS ARE COMPATIBLE >>
            equiv_prompt = base_equivalence_prompt.format(fact=fact, regen_question=question, user_question=user_question, *regen_answers)
            equiv_response = predict_w_log(equiv_prompt, 3)
            wait()

            binary_response = equiv_response.lower()[:10]

            if 'yes' in binary_response:
                e_results['uncertainty'][f'fact-{fidx}'][f'question-{qidx}'] = 0
            elif 'no' in binary_response:
                e_results['uncertainty'][f'fact-{fidx}'][f'question-{qidx}'] = 1
            else:
                # how to handle this?
                raise

        uncertainties = e_results['uncertainty'][f'fact-{fidx}'].values()
        log_w_indent(f'Final uncertainty for fact {fact}: {uncertainties}', 2)
        wait()

    # << ASSEMBLE FINAL RESPONSE >>
    log_w_indent('Final generation with uncertainty', 1)
    for fidx, fact in enumerate(facts):
        uncertainties =  e_results['uncertainty'].get(f'fact-{fidx}', {None: np.nan}).values()
        log_w_indent(f'(Uncertainty: {sum(uncertainties) / len(uncertainties)}) {fact}', 1)

2023-09-26 16:47:44 INFO     ********************************************************************************
2023-09-26 16:47:44 INFO      Starting with entity YG, user_question: "Who is Yarin Gal?"
2023-09-26 16:47:44 INFO     >>>> Input: Respond with four short sentences to the following question. Provide concrete facts rather than vague descriptions.
Who is Yarin Gal?
2023-09-26 16:47:48 INFO     >>>> Output: Yarin Gal is a highly accomplished computer scientist and researcher. He is currently a professor at the University of Oxford. He specializes in machine learning and artificial intelligence, with a particular focus on developing algorithms for probabilistic modeling and approximate inference. Yarin has published numerous papers in top-tier conferences and journals in the field, and his research has been widely cited by other experts in the domain.
2023-09-26 16:47:48 WARNING  Adding hallucination manually!
2023-09-26 16:47:48 INFO     >>>> Extracted facts: ['Yarin Gal is a hig

wait 


2023-09-26 16:50:39 INFO     >>>>>>>> Currently dealing with fact 0: Yarin Gal is a highly accomplished computer scientist and researcher.
2023-09-26 16:50:39 INFO     >>>>>>>>>>>> Input: In the context of the question "Who is Yarin Gal?" please generate a numbered list of two questions that might have the answer "Yarin Gal is a highly accomplished computer scientist and researcher."
Your questions should avoid using specific facts in the answer, but can be specific to the context implied by the original question. Your questions should be so specific that only one answer is possible. You can ask questions starting with "What", "Where", or "Who" if appropriate.
2023-09-26 16:50:41 INFO     >>>>>>>>>>>> Output: 1. What field does Yarin Gal specialize in?
2. What is one notable achievement of Yarin Gal in his career?
2023-09-26 16:50:41 INFO     >>>>>>>>>>>> Input: In the context of the question "Who is Yarin Gal?" please generate a numbered list of two questions that might have the answe

wait \


2023-09-26 16:50:49 INFO     >>>>>>>>>>>> Regenerate answers for question 0 "What field does Yarin Gal specialize in?":
2023-09-26 16:50:49 INFO     >>>>>>>>>>>>>>>> Input: In the context of "Who is Yarin Gal?" respond as concisely as possible to the following question: "What field does Yarin Gal specialize in?"
2023-09-26 16:50:50 INFO     >>>>>>>>>>>>>>>> Output: Machine learning and artificial intelligence.
2023-09-26 16:50:50 INFO     >>>>>>>>>>>>>>>> Input: In the context of "Who is Yarin Gal?" respond as concisely as possible to the following question: "What field does Yarin Gal specialize in?"
2023-09-26 16:50:51 INFO     >>>>>>>>>>>>>>>> Output: Machine learning and artificial intelligence.
2023-09-26 16:50:51 INFO     >>>>>>>>>>>>>>>> Input: In the context of "Who is Yarin Gal?" respond as concisely as possible to the following question: "What field does Yarin Gal specialize in?"
2023-09-26 16:50:51 INFO     >>>>>>>>>>>>>>>> Output: Machine learning and artificial intelligence

...

wait 


2023-09-26 16:51:27 INFO     >>>>>>>>>>>> Regenerate answers for question 1 "What is Yarin Gal's current occupation in the field of education?":
2023-09-26 16:51:27 INFO     >>>>>>>>>>>>>>>> Input: In the context of "Who is Yarin Gal?" respond as concisely as possible to the following question: "What is Yarin Gal's current occupation in the field of education?"
2023-09-26 16:51:28 INFO     >>>>>>>>>>>>>>>> Output: Yarin Gal's current occupation in education is assistant professor.
2023-09-26 16:51:28 INFO     >>>>>>>>>>>>>>>> Input: In the context of "Who is Yarin Gal?" respond as concisely as possible to the following question: "What is Yarin Gal's current occupation in the field of education?"
2023-09-26 16:51:29 INFO     >>>>>>>>>>>>>>>> Output: Yarin Gal's current occupation in the field of education is not known.
2023-09-26 16:51:29 INFO     >>>>>>>>>>>>>>>> Input: In the context of "Who is Yarin Gal?" respond as concisely as possible to the following question: "What is Yarin Gal'

wait 


2023-09-26 16:51:32 INFO     >>>>>>>>>>>> Regenerate answers for question 2 "What is Yarin Gal's current occupation at the University of Cambridge?":
2023-09-26 16:51:32 INFO     >>>>>>>>>>>>>>>> Input: In the context of "Who is Yarin Gal?" respond as concisely as possible to the following question: "What is Yarin Gal's current occupation at the University of Cambridge?"
2023-09-26 16:51:32 INFO     >>>>>>>>>>>>>>>> Output: Unknown.
2023-09-26 16:51:32 INFO     >>>>>>>>>>>>>>>> Input: In the context of "Who is Yarin Gal?" respond as concisely as possible to the following question: "What is Yarin Gal's current occupation at the University of Cambridge?"
2023-09-26 16:51:33 INFO     >>>>>>>>>>>>>>>> Output: Unknown.
2023-09-26 16:51:33 INFO     >>>>>>>>>>>>>>>> Input: In the context of "Who is Yarin Gal?" respond as concisely as possible to the following question: "What is Yarin Gal's current occupation at the University of Cambridge?"
2023-09-26 16:51:34 INFO     >>>>>>>>>>>>>>>> Output

KeyboardInterrupt: 